# DSA210 — Tech Job Market Salary Analysis
**Author:** cansatir  
**Datasets:** lukebarousse/data_jobs · BLS OEWS

## 0. Setup & Imports

In [1]:
from pathlib import Path
import ast, json, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Pin CWD to project root (works whether run interactively or via nbconvert)
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
os.chdir(PROJECT_ROOT)

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

DATA_RAW       = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
FIGURES        = PROJECT_ROOT / 'figures'
RESULTS        = PROJECT_ROOT / 'results'

for d in (DATA_PROCESSED, FIGURES, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')

Project root: /app


## 1. Load & Inspect Raw Datasets

In [2]:
# ── Jobs dataset ──────────────────────────────────────────────
df_jobs = pd.read_parquet(DATA_RAW / 'jobs_raw.parquet')
print(f'Shape: {df_jobs.shape}')
print(f'\nDtypes:\n{df_jobs.dtypes}')
print(f'\nMissing values (top 10):\n{df_jobs.isnull().sum().sort_values(ascending=False).head(10)}')
df_jobs.head()

Shape: (785741, 17)

Dtypes:
job_title_short           object
job_title                 object
job_location              object
job_via                   object
job_schedule_type         object
job_work_from_home          bool
search_location           object
job_posted_date           object
job_no_degree_mention       bool
job_health_insurance        bool
job_country               object
salary_rate               object
salary_year_avg          float64
salary_hour_avg          float64
company_name              object
job_skills                object
job_type_skills           object
dtype: object

Missing values (top 10):
salary_hour_avg      775079
salary_year_avg      763738
salary_rate          752674
job_type_skills      117037
job_skills           117037
job_schedule_type     12667
job_location           1045
job_country              49
company_name             18
job_via                   8
dtype: int64


,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills
0,Senior Data Engineer,Senior Clinical Data Engineer / Principal Clin...,"Watertown, CT",via Work Nearby,Full-time,False,"Texas, United States",2023-06-16 13:44:15,False,False,United States,None,NaN,NaN,Boehringer Ingelheim,None,None
1,Data Analyst,Data Analyst,"Guadalajara, Jalisco, Mexico",via BeBee México,Full-time,False,Mexico,2023-01-14 13:18:07,False,False,Mexico,None,NaN,NaN,Hewlett Packard Enterprise,"['r', 'python', 'sql', 'nosql', 'power bi', 't...","{'analyst_tools': ['power bi', 'tableau'], 'pr..."
2,Data Engineer,"Data Engineer/Scientist/Analyst, Mid or Senior...","Berlin, Germany",via LinkedIn,Full-time,False,Germany,2023-10-10 13:14:55,False,False,Germany,None,NaN,NaN,ALPHA Augmented Services,"['python', 'sql', 'c#', 'azure', 'airflow', 'd...","{'analyst_tools': ['dax'], 'cloud': ['azure'],..."
3,Data Engineer,LEAD ENGINEER - PRINCIPAL ANALYST - PRINCIPAL ...,"San Antonio, TX",via Diversity.com,Full-time,False,"Texas, United States",2023-07-04 13:01:41,True,False,United States,None,NaN,NaN,Southwest Research Institute,"['python', 'c++', 'java', 'matlab', 'aws', 'te...","{'cloud': ['aws'], 'libraries': ['tensorflow',..."
4,Data Engineer,Data Engineer- Sr Jobs,"Washington, DC",via Clearance Jobs,Full-time,False,Sudan,2023-08-07 14:29:36,False,False,Sudan,None,NaN,NaN,Kristina Daniel,"['bash', 'python', 'oracle', 'aws', 'ansible',...","{'cloud': ['oracle', 'aws'], 'other': ['ansibl..."


In [3]:
# ── OEWS enrichment dataset ───────────────────────────────────
OEWS_COLS = ['OCC_CODE', 'OCC_TITLE', 'A_MEAN', 'A_MEDIAN',
             'A_PCT10', 'A_PCT25', 'A_PCT75', 'A_PCT90']
oews_raw = pd.read_excel(DATA_RAW / 'oews_national.xlsx',
                         header=0, dtype={'OCC_CODE': str})
print(f'Shape: {oews_raw.shape}')
print(f'\nDtypes:\n{oews_raw.dtypes}')
print(f'\nMissing values (key cols):\n{oews_raw[OEWS_COLS].isnull().sum()}')
oews_raw[OEWS_COLS].head()

Shape: (1403, 32)

Dtypes:
AREA              int64
AREA_TITLE       object
AREA_TYPE         int64
PRIM_STATE       object
NAICS             int64
NAICS_TITLE      object
I_GROUP          object
OWN_CODE          int64
OCC_CODE         object
OCC_TITLE        object
O_GROUP          object
TOT_EMP           int64
EMP_PRSE        float64
JOBS_1000       float64
LOC_QUOTIENT    float64
PCT_TOTAL       float64
PCT_RPT         float64
H_MEAN           object
A_MEAN           object
MEAN_PRSE       float64
H_PCT10          object
H_PCT25          object
H_MEDIAN         object
H_PCT75          object
H_PCT90          object
A_PCT10          object
A_PCT25          object
A_MEDIAN         object
A_PCT75          object
A_PCT90          object
ANNUAL           object
HOURLY           object
dtype: object

Missing values (key cols):
OCC_CODE     0
OCC_TITLE    0
A_MEAN       0
A_MEDIAN     0
A_PCT10      0
A_PCT25      0
A_PCT75      0
A_PCT90      0
dtype: int64


,OCC_CODE,OCC_TITLE,A_MEAN,A_MEDIAN,A_PCT10,A_PCT25,A_PCT75,A_PCT90
0,00-0000,All Occupations,67920,49500,29990,36730,78810,125720
1,11-0000,Management Occupations,141760,122090,57010,79900,171610,#
2,11-1000,Top Executives,139860,104990,47510,68800,168490,#
3,11-1010,Chief Executives,262930,206420,73710,126080,#,#
4,11-1011,Chief Executives,262930,206420,73710,126080,#,#


In [4]:
print(f'Section 1 complete — jobs: {df_jobs.shape}, oews: {oews_raw.shape}')

Section 1 complete — jobs: (785741, 17), oews: (1403, 32)
